In [ ]:

hello world I love you and 

def candidate_penalty_ce_loss(model, sample, reduce=True, compute_custom_metrics=True):
    # net_output = model(**sample['net_input'])
    # target = model.get_targets(sample, net_output)
    batch_size, seq_length, vocab_size = 2, 7, 6
    net_output = torch.rand(batch_size, seq_length, vocab_size)
    net_output = net_output / net_output.norm(dim=-1, keepdim=True)
    target = torch.tensor([
        [2,3,1,5,2,3,4],
        [2,3,4,5,2,3,1]
    ])

    # nsentences = target.size(0)
    target = target.view(-1)

    # -- mle loss
    # lprobs = model.get_normalized_probs(net_output, log_probs=True)
    lprobs = torch.log(net_output)
    lprobs = lprobs.view(-1, lprobs.size(-1))
    true_token_lprobs = F.nll_loss(
        lprobs,
        target,
        ignore_index=self.padding_idx,
        reduction='none',
    )
    mle_loss = true_token_lprobs.sum()

    # -- custom loss
    # Maximize (1 - p(x_nt)) for negative target tokens x_nt (equivalently minimize -log(1-p(x_nt)))

    # - form negative targets
    with torch.no_grad():
        # E.g. DABCC | D | EFFGD => {A,B,C} are negative targets.
        if self.candidate_type == 'prev_context':
            # Make 'the triangle'.
            ctx_cands = target.unsqueeze(0).expand(target.size(0), target.size(0))
            ctx_cands_ = (ctx_cands.tril(-1) + self.padding_idx)
            ctx_cands_ = ctx_cands_ * ctx_cands_.triu()
            ctx_cands = ctx_cands.tril(-1) + ctx_cands_

            # Don't include the target for that timestep as a negative target.
            ctx_cands = ctx_cands.masked_fill(ctx_cands == target.unsqueeze(1), self.padding_idx)
            negative_targets = torch.zeros_like(lprobs).scatter_(1, ctx_cands, 1)
        else:
            raise NotImplementedError('candidate type %s' % self.candidate_type)

    # - compute loss
    one_minus_probs = torch.clamp((1.0 - lprobs.exp()), min=1e-5)

    custom_loss = -torch.log(one_minus_probs)*negative_targets
    custom_loss = custom_loss.sum()

    loss = mle_loss + self.rank_alpha * custom_loss

    # -- metrics
    logits = net_output[0].view(-1, net_output[0].size(-1))
    true_token_logits = -F.nll_loss(
        logits,
        target,
        ignore_index=self.padding_idx,
        reduction='none',
    )

    orig = utils.strip_pad(target, self.padding_idx)
    ntokens = orig.numel()
    sample_size = sample['target'].size(0) if self.args.sentence_avg else ntokens

    logging_output = {
        'custom_loss': utils.item(custom_loss.data),
        'loss': utils.item(mle_loss.data),
        'ntokens': ntokens,
        'nsentences': nsentences,
        'sample_size': sample_size,
    }
    if compute_custom_metrics:
        custom_output = TrainingMetrics.ranking_metrics(logits, true_token_logits, sample, ntokens, target)
        for k, v in custom_output.items():
            logging_output[k] = v

    return loss, sample_size, logging_output


In [ ]:
def sequence_penalty_loss(self, model, sample, reduce=True, generator=None):
    seq_len = sample['net_input']['src_tokens'].size(1)

    # make total number of tokens equal to the sequence length (for memory purposes)
    n_batches = seq_len // (self.sequence_prefix_length + self.sequence_completion_length)
    batch = batch_input_sequence_by_prefix_length(sample['net_input']['src_tokens'],
                                                prefix_length=self.sequence_prefix_length)
    batch = batch[:n_batches]

    pred_toks, lprobs = generator.generate_completion_greedy_training(model, batch,
                                                                    completion_length=self.sequence_completion_length)
    if self.sequence_candidate_type == 'repeat':
        mask = ngram_repeat_mask(pred_toks, self.sequence_ngram_n).type_as(lprobs)
    elif self.sequence_candidate_type == 'random':
        mask = torch.bernoulli(torch.zeros_like(pred_toks, dtype=torch.float).fill_(self.mask_p))

    pred_lprobs = lprobs.view(-1, lprobs.size(2)).gather(1, pred_toks.view(-1, 1))
    one_minus_probs = torch.clamp((1.0 - pred_lprobs.exp()), min=1e-20).view(pred_toks.size(0), pred_toks.size(1))
    loss = -torch.log(one_minus_probs)*mask
    loss = loss.sum()

    ntokens = pred_toks.numel()  # number of output tokens (tokens in completions)
    nsentences = batch.size(0)
    sample_size = ntokens
    logging_output = {
        'seq_loss': utils.item(loss.data),
        'seq_ntokens': ntokens,
        'seq_nsentences': nsentences,
        'seq_repeat_mask': utils.item(mask.sum().data),
        'seq_sample_size': sample_size,
    }

    # Sum each statistic, which will be normalized by the number of sentences in `aggregate_logging_outputs`.
    stats = defaultdict(float)
    for tok_list in pred_toks.cpu().tolist():
        ms = ngram_metrics(tok_list)
        for k, v in ms.items():
            stats[k] += v
    for k, v in stats.items():
        logging_output[k] = v

    return loss, sample_size, logging_output

## candidate penalty ce loss


In [22]:
import torch
import math
import torch.nn.functional as F

In [23]:
# net_output = model(**sample['net_input'])
# target = model.get_targets(sample, net_output)
batch_size, seq_length, vocab_size = 2, 7, 6
padding_idx = -100
rank_alpha = 1.0

net_output = torch.rand(batch_size, seq_length, vocab_size)

In [24]:
net_output = net_output / net_output.norm(dim=-1, keepdim=True)
target = torch.tensor([
    [2,3,1,5,2,3,4],
    [2,3,4,5,2,3,1]
])

# nsentences = target.size(0)
target = target.view(-1)

# -- mle loss
# lprobs = model.get_normalized_probs(net_output, log_probs=True)
lprobs = torch.log(net_output)
lprobs = lprobs.view(-1, lprobs.size(-1))
true_token_lprobs = F.nll_loss(
    lprobs,
    target,
    ignore_index=padding_idx,
    reduction='none',
)
mle_loss = true_token_lprobs.sum()


In [25]:
mle_loss

tensor(14.8161)

In [26]:
sum = lprobs[0][2] + lprobs[1][3] + lprobs[2][1] + lprobs[3][5] + lprobs[4][2] + lprobs[5][3] + lprobs[6][4] + \
        lprobs[7][2] + lprobs[8][3] + lprobs[9][4] + lprobs[10][5] + lprobs[11][2] + lprobs[12][3] + lprobs[13][1]

In [15]:
sum

tensor(-17.8938)

In [27]:
# Make 'the triangle'.
ctx_cands = target.unsqueeze(0).expand(target.size(0), target.size(0))
ctx_cands_ = (ctx_cands.tril(-1) + padding_idx)
ctx_cands_

tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99,  -95, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99,  -95,  -98, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99,  -95,  -98,  -97, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99,  -95,  -98,  -97,  -96, -100, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99,  -95,  -98,  -97,  -96,  -98, -100, -100, -100, -100,
         -100, -100],
        [ -98,  -97,  -99,  -95,  -98,  -97,  -96,  -98,  -97, -100, -100

In [28]:
ctx_cands_ = ctx_cands_ * ctx_cands_.triu()
ctx_cands_

tensor([[10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0,     0, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0,     0,     0, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0,     0,     0,     0, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0,     0,     0,     0,     0, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0,     0,     0,     0,     0,     0, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    0,     0,     0,     0,     0,     0,     0,     0,

In [29]:
ctx_cands = ctx_cands.tril(-1) + ctx_cands_
ctx_cands

tensor([[10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1,     5, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1,     5,     2, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1,     5,     2,     3, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1,     5,     2,     3,     4, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1,     5,     2,     3,     4,     2,

In [30]:

# Don't include the target for that timestep as a negative target.
ctx_cands = ctx_cands.masked_fill(ctx_cands == target.unsqueeze(1), padding_idx)
ctx_cands

tensor([[10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [ -100,     3,     1,     5, 10000, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,  -100,     1,     5,     2, 10000, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,     3,     1,     5,     2,     3, 10000, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [ -100,     3,     1,     5,  -100,     3,     4, 10000, 10000, 10000,
         10000, 10000, 10000, 10000],
        [    2,  -100,     1,     5,     2,  -100,     4,     2,

In [ ]:
negative_targets = torch.zeros_like(lprobs).scatter_(1, ctx_cands, 1)

In [ ]:
# # -- custom loss
# # Maximize (1 - p(x_nt)) for negative target tokens x_nt (equivalently minimize -log(1-p(x_nt)))

# # - form negative targets
# with torch.no_grad():
#     # E.g. DABCC | D | EFFGD => {A,B,C} are negative targets.
#     # if self.candidate_type == 'prev_context':
#     if True:
#         # Make 'the triangle'.
#         ctx_cands = target.unsqueeze(0).expand(target.size(0), target.size(0))
#         ctx_cands_ = (ctx_cands.tril(-1) + padding_idx)
#         ctx_cands_ = ctx_cands_ * ctx_cands_.triu()
#         ctx_cands = ctx_cands.tril(-1) + ctx_cands_

#         # Don't include the target for that timestep as a negative target.
#         ctx_cands = ctx_cands.masked_fill(ctx_cands == target.unsqueeze(1), padding_idx)
#         negative_targets = torch.zeros_like(lprobs).scatter_(1, ctx_cands, 1)
#     else:
#         raise NotImplementedError('candidate type %s' % self.candidate_type)

In [ ]:
# - compute loss
one_minus_probs = torch.clamp((1.0 - lprobs.exp()), min=1e-5)

custom_loss = -torch.log(one_minus_probs)*negative_targets
custom_loss = custom_loss.sum()

loss = mle_loss + rank_alpha * custom_loss


## my unlikelihood loss

In [31]:
import torch
import math
import torch.nn.functional as F

In [32]:
# net_output = model(**sample['net_input'])
# target = model.get_targets(sample, net_output)
batch_size, seq_length, vocab_size = 2, 7, 6
padding_idx = -100
rank_alpha = 1.0

net_output = torch.rand(batch_size, seq_length, vocab_size)

In [33]:
net_output = net_output / net_output.norm(dim=-1, keepdim=True)
target = torch.tensor([
    [2,3,1,5,2,3,4],
    [2,3,4,5,2,3,1]
])

# nsentences = target.size(0)
target = target.view(-1)

# -- mle loss
# lprobs = model.get_normalized_probs(net_output, log_probs=True)
lprobs = torch.log(net_output)
lprobs = lprobs.view(-1, lprobs.size(-1))
probs = torch.exp(lprobs)


In [34]:
ctx_cands = target.unsqueeze(0).expand(target.size(0), target.size(0))
# ctx_cands_ = (ctx_cands.tril(-1) + padding_idx)
# ctx_cands_ = ctx_cands_ * ctx_cands_.triu()
# ctx_cands = ctx_cands.tril(-1) + ctx_cands_
ctx_cands_ = torch.zeros_like(ctx_cands) + padding_idx
ctx_cands_

tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100

In [35]:
ctx_cands_ = ctx_cands_ - ctx_cands_.tril(-1)
ctx_cands_

tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0,    0, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0,    0,    0, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0,    0,    0,    0, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0,    0,    0,    0,    0, -100, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0,    0,    0,    0,    0,    0, -100, -100, -100, -100,
         -100, -100],
        [   0,    0,    0,    0,    0,    0,    0,    0,    0, -100, -100

In [36]:
ctx_cands = ctx_cands.tril(-1) + ctx_cands_
ctx_cands

tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1,    5, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1,    5,    2, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1,    5,    2,    3, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1,    5,    2,    3,    4, -100, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1,    5,    2,    3,    4,    2, -100, -100, -100, -100,
         -100, -100],
        [   2,    3,    1,    5,    2,    3,    4,    2,    3, -100, -100

In [37]:
mask = ctx_cands != padding_idx
valid_ctx_cands = ctx_cands[mask]

In [40]:
valid_indices = ctx_cands[mask]

In [42]:
row_indices = mask.nonzero(as_tuple=True)[0]
row_indices

tensor([ 1,  2,  2,  3,  3,  3,  4,  4,  4,  4,  5,  5,  5,  5,  5,  6,  6,  6,
         6,  6,  6,  7,  7,  7,  7,  7,  7,  7,  8,  8,  8,  8,  8,  8,  8,  8,
         9,  9,  9,  9,  9,  9,  9,  9,  9, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12,
        12, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
        13])

In [38]:
valid_ctx_cands

tensor([2, 2, 3, 2, 3, 1, 2, 3, 1, 5, 2, 3, 1, 5, 2, 2, 3, 1, 5, 2, 3, 2, 3, 1,
        5, 2, 3, 4, 2, 3, 1, 5, 2, 3, 4, 2, 2, 3, 1, 5, 2, 3, 4, 2, 3, 2, 3, 1,
        5, 2, 3, 4, 2, 3, 4, 2, 3, 1, 5, 2, 3, 4, 2, 3, 4, 5, 2, 3, 1, 5, 2, 3,
        4, 2, 3, 4, 5, 2, 2, 3, 1, 5, 2, 3, 4, 2, 3, 4, 5, 2, 3])

In [39]:
valid_ctx_cands.unsqueeze(1)

tensor([[2],
        [2],
        [3],
        [2],
        [3],
        [1],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [1],
        [5],
        [2],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [4],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [4],
        [2],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [4],
        [2],
        [3],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [4],
        [2],
        [3],
        [4],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [4],
        [2],
        [3],
        [4],
        [5],
        [2],
        [3],
        [1],
        [5],
        [2],
        [3],
        [4],
        [2],
        [3],
        [4],
        [5],

In [21]:
# Don't include the target for that timestep as a negative target.
ctx_cands = ctx_cands.masked_fill(ctx_cands == target.unsqueeze(1), padding_idx)
ctx_cands[ctx_cands==padding_idx] = 0
ctx_cands

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [2, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 3, 1, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [2, 0, 1, 5, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [2, 3, 1, 5, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 3, 1, 5, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0],
        [2, 0, 1, 5, 2, 0, 4, 2, 0, 0, 0, 0, 0, 0],
        [2, 3, 1, 5, 2, 3, 0, 2, 3, 0, 0, 0, 0, 0],
        [2, 3, 1, 0, 2, 3, 4, 2, 3, 4, 0, 0, 0, 0],
        [0, 3, 1, 5, 0, 3, 4, 0, 3, 4, 5, 0, 0, 0],
        [2, 0, 1, 5, 2, 0, 4, 2, 0, 4, 5, 2, 0, 0],
        [2, 3, 0, 5, 2, 3, 4, 2, 3, 4, 5, 2, 3, 0]])

In [ ]:

negative_targets = torch.zeros_like(lprobs).scatter_(1, ctx_cands, 1)